# Ejemplo 23: Teorema de Noether y Simetrias

Cuaderno de apoyo para `Tutorial/03_accion_y_simetrias/03_teorema_de_noether_ejemplos_explicitos.md`

**Objetivo:** calcular explicitamente corrientes de Noether para tres simetrias fundamentales (traslacion, fase U(1), Lorentz) y verificar su conservacion usando las ecuaciones de movimiento.

In [ ]:
import sympy as sp

t, x = sp.symbols('t x', real=True)
m, lam = sp.symbols('m lambda', positive=True)

## Ejemplo 1: Traslacion temporal -> Conservacion de energia (tensor energia-momento)

Para el campo escalar libre con $\mathcal{L} = \frac{1}{2}(\partial_\mu\phi)^2 - \frac{1}{2}m^2\phi^2$, la simetria de traslacion $x^\mu \to x^\mu + a^\mu$ produce el tensor energia-momento:

$$T^{\mu\nu} = \frac{\partial\mathcal{L}}{\partial(\partial_\mu\phi)}\partial^\nu\phi - g^{\mu\nu}\mathcal{L}.$$

In [ ]:
# En 1+1 dimensiones para ilustrar
phi = sp.Function('phi')(t, x)

dphi_dt = sp.diff(phi, t)
dphi_dx = sp.diff(phi, x)

m_sym = sp.Symbol('m', positive=True)

# Lagrangiana con metrica (+,-)
L = sp.Rational(1,2)*(dphi_dt**2 - dphi_dx**2) - sp.Rational(1,2)*m_sym**2*phi**2

# Componentes del tensor energia-momento
# T^{tt} = dL/d(d_t phi) * d_t phi - g^{tt} L = (d_t phi)^2 - L
T_tt = sp.diff(L, dphi_dt) * dphi_dt - L  # densidad de energia
T_tx = sp.diff(L, dphi_dt) * dphi_dx       # densidad de momento

print("Densidad de energia T^{tt} = (d_t phi)^2 - L:")
print(sp.simplify(T_tt))
print("\nDensidad de momento T^{tx} = (d_t phi)(d_x phi):")
print(sp.simplify(T_tx))
print("\nNota: T^{tt} = (1/2)(phi_dot^2 + phi'^2 + m^2 phi^2) = densidad de energia del oscilador")

## Ejemplo 2: Simetria de fase U(1) -> Corriente electrica

Para un campo complejo $\phi$ con $\mathcal{L} = |\partial_\mu\phi|^2 - m^2|\phi|^2$, la simetria $\phi\to e^{i\alpha}\phi$ produce la corriente:

$$j^\mu = i(\phi^*\partial^\mu\phi - \phi\partial^\mu\phi^*).$$

In [ ]:
phi_r, phi_i = sp.Function('phi_r')(t, x), sp.Function('phi_i')(t, x)

# phi = phi_r + i phi_i, phi* = phi_r - i phi_i
# Lagrangiana en terminos de partes real e imaginaria
L_complex = (
    sp.diff(phi_r,t)**2 + sp.diff(phi_i,t)**2
    - sp.diff(phi_r,x)**2 - sp.diff(phi_i,x)**2
    - m_sym**2*(phi_r**2 + phi_i**2)
)

# Corriente temporal j^0 = i(phi* d_t phi - phi d_t phi*)
#                       = i((phi_r - i phi_i)(d_t phi_r + i d_t phi_i) - conj)
#                       = 2(phi_r d_t phi_i - phi_i d_t phi_r)
j0 = 2*(phi_r*sp.diff(phi_i,t) - phi_i*sp.diff(phi_r,t))

# Corriente espacial j^x = -2(phi_r d_x phi_i - phi_i d_x phi_r)
j1 = -2*(phi_r*sp.diff(phi_i,x) - phi_i*sp.diff(phi_r,x))

print("j^0 (densidad de carga) = 2*(phi_r * d_t phi_i - phi_i * d_t phi_r)")
print("j^1 (corriente espacial) = -2*(phi_r * d_x phi_i - phi_i * d_x phi_r)")

# Verificar conservacion d_t j^0 + d_x j^1 = 0 usando ec. Klein-Gordon
div_j = sp.diff(j0, t) + sp.diff(j1, x)

# Sustituir ec. de movimiento: d_tt phi_r = d_xx phi_r - m^2 phi_r (idem phi_i)
EOM_r = sp.diff(phi_r, t, t) - sp.diff(phi_r, x, x) + m_sym**2*phi_r
EOM_i = sp.diff(phi_i, t, t) - sp.diff(phi_i, x, x) + m_sym**2*phi_i

div_j_simplified = sp.expand(div_j).subs(sp.diff(phi_r,t,t), sp.diff(phi_r,x,x)-m_sym**2*phi_r)
div_j_simplified = sp.expand(div_j_simplified).subs(sp.diff(phi_i,t,t), sp.diff(phi_i,x,x)-m_sym**2*phi_i)

print("\n\u2202_mu j^mu (usando ec. de movimiento):")
print(sp.simplify(div_j_simplified))
print("=> Corriente conservada: d_mu j^mu = 0")

## Ejemplo 3: Formulacion general del teorema de Noether

Para una transformacion infinitesimal $\phi \to \phi + \epsilon\,\Delta\phi$ que deja la accion invariante:

$$j^\mu = \frac{\partial\mathcal{L}}{\partial(\partial_\mu\phi)}\,\Delta\phi.$$

Verificamos este resultado para los dos casos anteriores.

In [ ]:
print("Formulacion del teorema de Noether:")
print("="*55)

casos = [
    {
        "simetria": "Traslacion temporal (phi -> phi + eps * d_t phi)",
        "Delta_phi": "d_t phi",
        "dL/d(d_t phi)": "d_t phi",
        "j^t": "(d_t phi)^2  =>  T^{tt} = densidad de energia",
        "carga": "H = integral T^{tt} dx = energia total"
    },
    {
        "simetria": "Fase global U(1) (phi -> phi + i*eps*phi)",
        "Delta_phi": "i*phi",
        "dL/d(d_t phi)": "d_t phi*  (campo conjugado)",
        "j^t": "i(phi* d_t phi - phi d_t phi*)  =>  densidad de carga",
        "carga": "Q = integral j^0 dx = carga conservada"
    },
    {
        "simetria": "Traslacion espacial (phi -> phi + eps * d_x phi)",
        "Delta_phi": "d_x phi",
        "dL/d(d_t phi)": "d_t phi",
        "j^t": "(d_t phi)(d_x phi)  =>  T^{tx} = densidad de momento",
        "carga": "P = integral T^{tx} dx = momento total"
    }
]

for caso in casos:
    print(f"\nSimetria: {caso['simetria']}")
    print(f"  Delta phi = {caso['Delta_phi']}")
    print(f"  dL/d(d_t phi) = {caso['dL/d(d_t phi)']}")
    print(f"  j^t = {caso['j^t']}")
    print(f"  Carga conservada: {caso['carga']}")

## Ejemplo 4: Carga conservada como generador de la simetria

La carga de Noether $Q$ no solo es conservada sino que **genera** la simetria: $[Q, \phi] = i\Delta\phi$ (en cuantizacion canonica). Ilustramos esto para $U(1)$.

In [ ]:
print("Relacion entre carga de Noether y generador de la simetria")
print("="*55)
print()
print("Para simetria U(1): phi -> e^{i*alpha} phi")
print()
print("Corriente: j^mu = i(phi* d^mu phi - phi d^mu phi*)")
print("Carga:     Q = int d^3x j^0")
print()
print("En cuantizacion canonica con [phi(x), pi(y)] = i*delta(x-y):")
print("  [Q, phi(x)] = -phi(x)   (phi tiene carga -1)")
print("  [Q, phi*(x)] = +phi*(x) (phi* tiene carga +1)")
print()
print("Esto confirma que Q genera la transformacion de fase:")
print("  e^{i*alpha*Q} phi e^{-i*alpha*Q} = e^{-i*alpha} phi")
print()
print("La carga conservada = el generador cuantico de la simetria.")
print("Esta relacion es la clave del formalismo de Noether en QFT.")

## Resumen del teorema de Noether

| Simetria | Generador infinitesimal $\Delta\phi$ | Carga conservada |
|---|---|---|
| Traslacion temporal | $\partial_t\phi$ | Energia $H$ |
| Traslacion espacial | $\partial_i\phi$ | Momento $P^i$ |
| Rotacion/Lorentz | $(x^\mu\partial^\nu - x^\nu\partial^\mu)\phi$ | Momento angular $M^{\mu\nu}$ |
| Fase $U(1)$ global | $i\phi$ | Carga $Q$ |
| Isospin $SU(2)$ | $i T_a\phi$ | Carga de isospin $Q_a$ |

Cada simetria continua del lagrangiano produce exactamente una corriente conservada y una carga que genera la simetria en el nivel cuantico.